# 📘 Guide 04: Technical foundations

**Who it's for:** you've heard "set an environment variable", "call the API" or "it returns JSON" and nodded along. **Time:** about 25 minutes.

These four ideas show up in every single lab. Once they click, the rest of the course gets much easier.

### 🎯 You'll learn
1. Environment variables and the `.env` file
2. What an API is, and what HTTP requests and responses look like
3. JSON: the language APIs speak
4. Virtual environments and why `uv run` matters

## Part 1: Environment variables and `.env`

Your OpenAI key must never appear in a notebook or a script: you'd leak it the moment you shared your code. Instead it lives in a file called `.env` at the repo root:

```
OPENAI_API_KEY=sk-proj-...
CHAT_MODEL=openai:gpt-5.4-mini
BUDGET_MODE=false
```

`load_dotenv()` copies those lines into the process's environment, and code reads them with `os.getenv`. Here's the same thing with a fake, throw-away `.env` so you can see it working:

In [ ]:
import os
import tempfile
from pathlib import Path

from dotenv import load_dotenv

demo_env = Path(tempfile.mkdtemp()) / ".env"
demo_env.write_text("GRADEWISE_DEMO_KEY=sk-demo-1234567890\nDEMO_BUDGET_MODE=true\n", encoding="utf-8")

load_dotenv(demo_env, override=True)

key = os.getenv("GRADEWISE_DEMO_KEY")
print(f"✅ Key loaded, begins {key[:7]}...  (💡 only ever print a prefix)")
print("Budget mode as text:", repr(os.getenv("DEMO_BUDGET_MODE")))
print("Missing variable:   ", os.getenv("NOT_SET", "default value"))

Notice that environment variables are always **strings**: `"true"`, not `True`. That's why GradeWise uses **pydantic-settings** in `gradewise/config.py`: it reads `.env` once and converts values to proper types (see Guide 10).

Setting a variable in the terminal works too, for that session only:
- Windows (PowerShell): `$env:BUDGET_MODE = "true"`
- Mac / Linux: `export BUDGET_MODE=true`

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/important.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important</h2>
            <span style="color:#ff7800;">The most common setup bug in any course: editing <code>.env</code> and forgetting to <b>save</b> it. The second most common: naming it <code>.env.txt</code> by accident on Windows. Turn on <i>File name extensions</i> in Explorer's View menu to check.</span>
        </td>
    </tr>
</table>

## Part 2: APIs and HTTP

An **API** is a door into someone else's program. OpenAI's API lets your code send a prompt and receive an answer. Almost all web APIs use **HTTP**, the same protocol your browser uses:

- A **request** has a *method* (`GET` to read, `POST` to send data), a *URL*, *headers* (like `Authorization: Bearer <your key>`) and often a *body*.
- A **response** has a *status code* (`200` OK, `401` bad key, `429` rate-limited, `500` server error), headers, and a body.

We can watch a full request/response round trip without the internet, using `httpx` with a fake ("mock") server:

In [ ]:
import httpx


def fake_grading_server(request: httpx.Request) -> httpx.Response:
    # 💡 Pretend to be a web API: check the key, then answer with JSON
    if request.headers.get("Authorization") != "Bearer sk-demo":
        return httpx.Response(401, json={"error": "Invalid API key"})
    answer = request.read().decode()
    return httpx.Response(200, json={"score": 4, "out_of": 5, "answer_length": len(answer)})


client = httpx.Client(transport=httpx.MockTransport(fake_grading_server), base_url="https://api.example.com")

good = client.post("/grade", headers={"Authorization": "Bearer sk-demo"}, content="A stack is LIFO...")
bad = client.post("/grade", headers={"Authorization": "Bearer wrong"}, content="...")

print("Good request →", good.status_code, good.json())
print("Bad key      →", bad.status_code, bad.json())

In the course you'll rarely write raw HTTP: LangChain's `init_chat_model` does it for you. But when you see `401` or `429` in an error, you'll now know exactly what happened.

## Part 3: JSON

**JSON** is plain text that represents data: objects `{}`, lists `[]`, strings, numbers, `true` / `false` / `null`. It maps almost one-to-one onto Python dictionaries and lists:

In [ ]:
import json

grade = {"student": "Sam", "question": "Q2", "marks": 3.5, "max_marks": 5, "flags": [], "moderated": False}

text = json.dumps(grade, indent=2)  # Python → JSON text
print(text)

back = json.loads(text)  # JSON text → Python
print("\nType after loading:", type(back).__name__, "| moderated is", back["moderated"])

Watch the small differences: Python's `False` and `None` become JSON's `false` and `null`. LLMs are very good at producing JSON, which is how GradeWise gets structured marks back from a model (week 1, day 3).

## Part 4: Virtual environments and `uv`

Each project needs its own set of packages at specific versions. A **virtual environment** (the `.venv` folder) is a private Python installation just for this repo.

- `uv sync` reads `pyproject.toml` + `uv.lock` and makes `.venv` match exactly.
- `uv run <command>` runs the command using that `.venv`, without you having to "activate" anything.
- In a notebook, the **kernel** (top-right) must be the `.venv` one.

Let's confirm which Python this notebook uses:

In [ ]:
import sys

print("Python:", sys.version.split()[0])
print("From:  ", sys.executable)
print("✅ Using the project .venv" if ".venv" in sys.executable else "⚠️ Not the .venv kernel: see setup/SETUP-*.md")

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/exercise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f59e0b;">Try it</h2>
            <span style="color:#f59e0b;">Modify the fake server in Part 2 so it returns <code>429</code> with <code>{"error": "Slow down"}</code> when the answer is longer than 50 characters. Then write a JSON string by hand for a grade with a <code>null</code> comment, and load it with <code>json.loads</code>.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/resources.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#3b82f6;">Resources</h2>
            <span style="color:#3b82f6;">• MDN: An overview of HTTP: <a href="https://developer.mozilla.org/en-US/docs/Web/HTTP/Overview">developer.mozilla.org/en-US/docs/Web/HTTP/Overview</a><br/>• HTTP status codes: <a href="https://developer.mozilla.org/en-US/docs/Web/HTTP/Status">developer.mozilla.org/en-US/docs/Web/HTTP/Status</a><br/>• python-dotenv: <a href="https://github.com/theskumar/python-dotenv">github.com/theskumar/python-dotenv</a><br/>• httpx: <a href="https://www.python-httpx.org/">www.python-httpx.org/</a></span>
        </td>
    </tr>
</table>